# Step 1 - Prototype 1

In this small example, we only want to formulate the problem of 2 trains going from the same starting station to the same destination station on a straight line of tracks. The first half should have parallel tracks, while the second one only has one set of tracks. That there could be an optimization, both schedules should overlap. However, for this formulation approach even this is incidental.

For this, we will chop up time and space into discrete intervals (so we only boolean / integer variables). To treat trains with different speeds in this formalism, we'll have to either go more fine-grained or reformulate). The optimizer can then place trains on the tracks and 'let them move or wait' by imposing rules and 'flow conservation'.

**Idea:**
Actually, to make this example already really optimizable, we could say that one of the trains can move more than one (e.g., 2) blocks per time step, but make the slower one go first (by that much, that the faster train would naturally surpass the slower one on the capacity-1-segment)... We can either iterate on this example after formulating the super basic one or create a new file and make some tweaks. This is probably the better idea...

## Implementation Analysis

- We have successfully implemented the task as an MILP
- We can build a straight track with blocks of different capacities. Here we exceeded the goal and can already use $n$ segments (block series of different capacities). However, we only have a start and destination station and can't add more in the middle. For this we need to change the implementation (and think about how to best do this).
- Technically, we can already use $n$ trains as well. Here, we can feel the issue of not having real train stations again though (we currently simulate it via 'high-capacity length-one' blocks).
- The optimization goal should be adjusted. This should help with lazy-solving issues (a train might not be moved at all if it can't reach the destination in the given time horizon). Also, this will be imperative for more complex optimization goals later on (like reducing dwell times and which that the network efficiency).

$\implies$ Essentially, I would say this qualifies as completing step 1. However, since there are still a few wrinkles and possibly foundational implementations (or decisions thereof) we should improve 'soonish' we can create a second prototype to test those on an easier codebase.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Initialize the Model
model = gp.Model("Basic_MILP")

In [ ]:
# Define the track system and timetable

# Track system data (store number of blocks b_i per segment of tracks with capacities c_i)
segment_lengths = [5, 5, 2]
capacities = [2, 1, 3]

# Timetable data
time_horizon = 20

leave_times = [2, 3, 5]    # Due to the implementation ('looking-back') leave_times_i has to be >= 1
arrival_times = [18, 19, 20]

# Deriving some additional variables
num_trains = len(leave_times)
num_blocks = sum(segment_lengths)

# Helper list for capacity references
capacities_list = np.repeat(capacities, segment_lengths)

In [ ]:
# Define Decision Variables

# To describe our train movement we need a way to know which train is where at what
# time (-> 3D time-position grid of boolean values)

x = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x")

In [ ]:
# General Constraints

# Uniqueness constraint for trains to only exist once on a block (on only a single track at a given time)
# Additionally, we tackle the spawning (leave times), as before a train does not exist on any block
for i in range(num_trains):
    for k in range(time_horizon):

        # Sum over all blocks j to ensure that a train only exists once on this block
        active_tracks = gp.quicksum(x[i, j, k] for j in range(num_blocks))

        # If the train hasn't spawned (or left) yet, it can't exist on the track and we enforce this
        if k < leave_times[i]:
            model.addConstr(active_tracks == 0, name=f"NotSpawned_Train{i}_Time{k}")
        else:
            # If it already exists, we enforce that the train only exists on one track at any time k
            model.addConstr(active_tracks == 1, name=f"UniquePosition_Train{i}_Time{k}")

In [ ]:
# Capacity constraints (the sum over all tracks j and all trains i <= capacities_i)
for k in range(time_horizon):

    # Here we want to sum over all i AND j and add the constraint that this should be smaller than
    # Is this formulation correct and can this be made more compact (yes: x.sum('*', j, k))?
    for j in range(num_blocks):
        occupied_tracks = gp.quicksum(x[i, j, k] for i in range(num_trains))

        # Add constraint to enforce capacity maximum across block j
        model.addConstr(occupied_tracks <= capacities_list[j], name=f"Capacity_Block{j}_Time{k}")

In [ ]:
# Movement Constraints

# A train can (after spawning) either wait or move
for i in range(num_trains):

    # At every time point, we need to differentiate between different scenarios
    for k in range(time_horizon):

        # If the train hasn't left yet, it's not yet on the grid and the spawn constraint takes care of this
        if k < leave_times[i]:
            pass
        # Here the train spawns, and we fix a position (at the very beginning of the tracks (j = 0))
        elif k == leave_times[i]:
            model.addConstr(x[i, 0, k] == 1, name=f"Spawn_Train{i}_Time{k}_on_Block0")
        else:

            # Handle out back-looking approach
            for j in range(num_blocks):

                # If the train is on block one, it must have already been there (j is always ≠ -1)
                if j == 0:
                    model.addConstr(x[i, 0, k] <= x[i, 0, k - 1], name=f"Move_Train{i}_Time{k}_Block0")
                else:
                    # General back-look constraint (train must have been here or in the previous block)
                    model.addConstr(x[i, j, k] <= x[i, j, k - 1] + x[i, j - 1, k - 1], name=f"Move_Train{i}_Time{k}_Block{j}")

In [ ]:
# Set the Model Objective

# Here we basically only want to minimize the final delay (actual_arrival_time - timetable_arrival_time)

# Now though, just for testing purposes, we will choose a simpler to implement but later 'stupid' goal:
# Maximize the time spent at the destination.

last_block = num_blocks - 1

objective = gp.quicksum(x[i, last_block, k] for i in range(num_trains) for k in range(time_horizon))
model.setObjective(objective, GRB.MAXIMIZE)

In [ ]:
# Run the optimization
model.optimize()

# Code by Gemini

# Print the results if an optimal solution is found
if model.Status == GRB.OPTIMAL:
    print("\n" + "="*30)
    print("🚂 OPTIMAL SCHEDULE FOUND 🚂")
    print("="*30)

    for i in range(num_trains):
        print(f"\n--- Train {i} ---")
        for k in range(time_horizon):
            for j in range(num_blocks):
                # .X extracts the solved value from the Gurobi variable.
                # We check > 0.5 to account for minor floating-point tolerances in the solver.
                if x[i, j, k].X > 0.5:
                    print(f"Time {k:2d} -> Block {j}")
else:
    print("\n❌ No optimal solution found. The model might be Infeasible.")